questo blocco di codice serve a importare tutti gli strumenti necessari per costruire e addestrare una rete neurale per la computer vision.

In [2]:
import torch
# Importa la libreria principale PyTorch che fornisce funzionalità base per il deep learning come tensori, autograd, e operazioni su GPU

from torch import nn
# from torch: dal pacchetto principale di PyTorch
# import nn: importa il modulo "nn" (neural networks) che contiene classi per costruire reti neurali come layer, funzioni di attivazione e contenitori sequenziali

from torch.utils.data import DataLoader
# from torch.utils.data: dal sottopacchetto "utils.data" di PyTorch
# - utils: è il sottopacchetto che contiene strumenti di utilità generale
# - data: è il modulo specifico per la gestione dei dati
# import DataLoader: importa la classe DataLoader che serve per caricare i dati in batch durante l'addestramento

from torchvision import datasets
# from torchvision: dal pacchetto torchvision, dedicato alla computer vision
# import datasets: importa il modulo che contiene dataset di immagini predefiniti e pronti all'uso come MNIST, CIFAR10, ImageNet

from torchvision.transforms import ToTensor
# from torchvision.transforms: dal sottopacchetto "transforms" di torchvision
# - transforms: contiene funzioni per trasformare e preprocessare immagini
# import ToTensor: importa la trasformazione specifica che converte immagini in formato PIL o array numpy in tensori PyTorch



Questo codice scarica il dataset Fashion-MNIST (immagini di vestiti, scarpe, borse in bianco e nero 28x28 pixel) e lo divide in:

training_data: 60.000 immagini per addestrare il modello

test_data: 10.000 immagini per testare il modello

I dati vengono salvati nella cartella "data" e convertiti automaticamente in tensori PyTorch

In [3]:
# Download training data from open datasets.

training_data = datasets.FashionMNIST(
    # datasets.FashionMNIST: dal modulo torchvision.datasets, carica il dataset Fashion-MNIST (immagini di abbigliamento)

    root="data",
    # root: specifica la cartella dove salvare/scaricare i dati
    # "data": crea una cartella chiamata "data" nella directory corrente

    train=True,
    # train=True: carica la parte di TRAINING del dataset (60.000 immagini)
    # train=False: avrebbe caricato la parte di TEST (10.000 immagini)

    download=True,
    # download=True: scarica il dataset da internet se non è già presente nella cartella "data"

    transform=ToTensor(),
    # transform=ToTensor(): applica la trasformazione ToTensor() a ogni immagine
    # Converte le immagini PIL in tensori PyTorch e normalizza i pixel da [0,255] a [0.0,1.0]
)


# Download test data from open datasets.

test_data = datasets.FashionMNIST(
    # datasets.FashionMNIST: stesso dataset, ma per il TEST

    root="data",
    # root: stessa cartella "data" (così training e test sono insieme)

    train=False,
    # train=False: carica la parte di TEST (10.000 immagini)

    download=True,
    # download=True: scarica anche questi (di solito vengono insieme, ma meglio metterlo)

    transform=ToTensor(),
    # transform=ToTensor(): stessa trasformazione applicata ai dati di test
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 14.9MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 303kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.64MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 21.4MB/s]


Prepara i dati per l'addestramento:

- Crea due DataLoader: uno per il training e uno per il test
- Imposta batch_size=64, quindi ogni batch conterrà 64 immagini con le relative etichette
- Il ciclo for finale mostra la struttura del primo batch per verificare che tutto funzioni

In [4]:
batch_size = 64
# batch_size = 64: imposta a 64 il numero di campioni elaborati insieme in un singolo batch

# Create data loaders.

train_dataloader = DataLoader(training_data, batch_size=batch_size)
# train_dataloader: crea un DataLoader per i dati di addestramento
# DataLoader(training_data, batch_size=batch_size): prende il dataset training_data e lo configura per restituire batch da 64 elementi

test_dataloader = DataLoader(test_data, batch_size=batch_size)
# test_dataloader: crea un DataLoader per i dati di test
# DataLoader(test_data, batch_size=batch_size): stesso batch_size per i dati di test

for X, y in test_dataloader:
    # for X, y in test_dataloader: itera sul dataloader di test (prende solo il primo batch grazie a break)
    # X: contiene le immagini del batch (64 immagini)
    # y: contiene le etichette corrispondenti alle 64 immagini

    print(f"Shape of X [N, C, H, W]: {X.shape}")
    # print: mostra la forma del tensore X
    # [N, C, H, W]: formato standard per immagini in PyTorch
    # N = numero di campioni nel batch (64)
    # C = canali (1 per Fashion-MNIST perché sono in bianco e nero)
    # H = altezza dell'immagine (28 pixel)
    # W = larghezza dell'immagine (28 pixel)

    print(f"Shape of y: {y.shape} {y.dtype}")
    # print: mostra la forma del tensore y e il tipo di dato
    # y.shape: sarà [64] (un'etichetta per ogni immagine)
    # y.dtype: intero (le etichette sono numeri interi da 0 a 9)

    break
    # break: esce dal ciclo dopo il primo batch (serve solo per vedere com'è fatto, non processa tutto)

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


**Creating Models** :

fa vedere come si crea la rete defindendo classi, moduli e layer che i dati subiranno, crea la rete

In [5]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device: determina se usare GPU o CPU per i calcoli
# torch.accelerator.current_accelerator().type: rileva il tipo di acceleratore disponibile (GPU, TPU, ecc.)
# if torch.accelerator.is_available(): verifica se c'è un acceleratore disponibile
# else "cpu": se non c'è acceleratore, usa la CPU
# Il risultato sarà "cuda" (GPU NVIDIA), "tpu" (TPU Google) o "cpu"

print(f"Using {device} device")
# print: mostra quale dispositivo viene utilizzato (es. "Using cuda device")



# Define model

class NeuralNetwork(nn.Module):
    # class NeuralNetwork: definisce una nuova classe chiamata NeuralNetwork
    # (nn.Module): eredita dalla classe Module di PyTorch (classe base per tutte le reti neurali)

    def __init__(self):
        # __init__: metodo costruttore, viene eseguito quando crei un'istanza della classe
        # self si riferirà all'oggetto che verrà creato, è il modo in cui un oggetto fa riferimento a se stesso.
        super().__init__()
        # super().__init__(): chiama il costruttore della classe padre (nn.Module)

        self.flatten = nn.Flatten()
        # self.flatten: attributo che contiene un layer Flatten
        # nn.Flatten(): layer che appiattisce immagini 2D (28x28) in un vettore 1D (784)

        self.linear_relu_stack = nn.Sequential(
            # self.linear_relu_stack: attributo che contiene una sequenza di layer
            # nn.Sequential(): contenitore che applica i layer in ordine
            nn.Linear(28*28, 512),
            # nn.Linear(28*28, 512): strato lineare che va da 784 neuroni a 512 neuroni
            nn.ReLU(),
            # nn.ReLU(): funzione di attivazione ReLU (rettificatore lineare)
            nn.Linear(512, 512),
            # nn.Linear(512, 512): secondo strato lineare da 512 a 512 neuroni
            nn.ReLU(),
            # nn.ReLU(): altra attivazione ReLU
            nn.Linear(512, 10)
            # nn.Linear(512, 10): strato finale da 512 a 10 neuroni (10 classi di output)
        )

        #in pratica self.linear_relu_stack prende la struttura di nn.Sequential in nn e la rinomina,
        #in sostanza cambiando delle parti tipo relu e linear dando un ordine a come la funzione diciamo self.linear_relu_stack
        #elaborerà i dati


        # TEORIA DELLA RETE NEURALE:

        # INPUT: 784 neuroni (immagine 28x28 pixel appiattita, uno per ogni pixel)
        # Ogni pixel è un neurone con valore da 0 a 1 (intensità)
        #
        # STRATO 1: nn.Linear(784, 512)
        # - Connette ogni pixel a 512 neuroni nascosti (512 perche potenze di 2)
        # - Impara pattern semplici: bordi, angoli, texture
        #
        # ATTIVAZIONE 1: nn.ReLU()
        # - Funzione non lineare: f(x) = max(0, x)
        # - Azzera i valori negativi, introduce non-linearità: attiva o meno il neurone
        #
        # STRATO 2: nn.Linear(512, 512)
        # - Connette 512 neuroni ad altri 512 neuroni
        # - Impara pattern complessi: maniche, colli, tasche
        #
        # ATTIVAZIONE 2: nn.ReLU()
        # - Ancora non-linearità
        #
        # OUTPUT: nn.Linear(512, 10)
        # - 10 neuroni, uno per ogni classe di abbigliamento
        # - Ogni neurone rappresenta una probabilità per quella classe
        #
        # CLASSI FASHION-MNIST:
        # 0: T-shirt/top | 1: Pantaloni  | 2: Pullover | 3: Vestito
        # 4: Cappotto    | 5: Sandali    | 6: Camicia   | 7: Sneaker
        # 8: Borsa       | 9: Stivaletti
        #
        # La rete impara in modo gerarchico:
        # pixel → pattern semplici → pattern complessi → classificazione


    def forward(self, x):
        # forward: metodo definito da nn.Module, esegue il passaggio in avanti della rete
        # self: riferimento all'istanza corrente
        # x: input (le immagini in ingresso)

        x = self.flatten(x)
        # x = self.flatten(x): applica il layer flatten a x (appiattisce le immagini)

        logits = self.linear_relu_stack(x)
        # logits = self.linear_relu_stack(x): passa x attraverso la sequenza di layer lineari e ReLU
        # logits: output grezzi (non normalizzati) prima dell'applicazione di softmax

        return logits
        # return logits: restituisce i logits come output della rete



model = NeuralNetwork().to(device)
# model = NeuralNetwork(): crea un'istanza (oggetto) della classe NeuralNetwork
# .to(device): sposta il modello sul dispositivo specificato (GPU o CPU)
# model: variabile che contiene il modello pronto per l'addestramento

print(model)
# print(model): stampa la struttura del modello (mostra tutti i layer)

Using cpu device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


Optimizing the Model Parameters:

da il via al modello, definisce loss, gradienti e come si aggiornano i pesi per il TRAINING


In [7]:
def train(dataloader, model, loss_fn, optimizer):
    # train: funzione che esegue un'epoca di addestramento
    # dataloader: contiene i dati di training in batch
    # model: la rete neurale da addestrare
    # loss_fn: funzione per calcolare l'errore (loss function)
    # optimizer: algoritmo che aggiorna i pesi (es. SGD, Adam)

    size = len(dataloader.dataset)
    # size: numero totale di campioni nel dataset (es. 60.000 per Fashion-MNIST)

    model.train()
    # model.train(): mette il modello in modalità training
    # Attiva dropout, batch normalization, e dice a PyTorch che stiamo addestrando

    # DROPOUT E BATCH NORMALIZATION:
    #
    # DROPOUT (nn.Dropout):
    # - Durante training: disattiva casualmente alcuni neuroni
    # - Durante test: si disattiva, usa tutti i neuroni
    # - Scopo: evita che la rete dipenda da pochi neuroni, riduce overfitting
    #
    # BATCH NORMALIZATION (nn.BatchNorm):
    # - Durante training: normalizza i dati del batch (media=0, varianza=1)
    # - Durante test: usa le statistiche imparate durante training
    # - Scopo: stabilizza l'addestramento, accelera convergenza
    #
    # model.train(): ATTIVA dropout, batch norm si adatta al batch
    # model.eval():  DISATTIVA dropout, batch norm usa statistiche fisse



    for batch, (X, y) in enumerate(dataloader):
        # batch: indice del batch corrente (0, 1, 2, ...)
        # (X, y): un batch di dati (X = immagini, y = etichette)
        # enumerate: dà sia l'indice che i dati

        X, y = X.to(device), y.to(device)
        # X.to(device): sposta le immagini sul dispositivo (GPU/CPU)
        # y.to(device): sposta le etichette sul dispositivo



        # Compute prediction error


        pred = model(X)
        # pred = model(X): passa le immagini al modello, ottieni le predizioni
        # model(X) chiama automaticamente il metodo forward della classe

        # METODO FORWARD: è il percorso che i dati fanno dentro la rete neurale.
        # Definisce il percorso dei dati attraverso la rete
        # Chiamato automaticamente quando fai model(X)
        #
        # 1. x = self.flatten(x)
        #    Appiattisce l'immagine 28x28 in un vettore di 784 pixel
        #
        # 2. logits = self.linear_relu_stack(x)
        #    Passa il vettore attraverso i layer lineari e ReLU:
        #    - Linear(784 → 512): primo strato denso
        #    - ReLU: attivazione non lineare
        #    - Linear(512 → 512): secondo strato denso
        #    - ReLU: attivazione non lineare
        #    - Linear(512 → 10): strato finale per le 10 classi
        #
        # 3. return logits
        #    Restituisce 10 valori (logits) per ogni immagine
        #    Valori più alti = maggiore probabilità per quella classe



        loss = loss_fn(pred, y)
        # loss = loss_fn(pred, y): calcola l'errore tra predizioni e verità
        # loss_fn è la funzione di perdita (es. CrossEntropyLoss), credo sia ancora da definire

        # Backpropagation --- retropropagazione dell'errore



        loss.backward()

        # loss.backward(): calcola i gradienti per tutti i parametri
        # PyTorch automaticamente calcola derivate rispetto a ogni peso
        #
        # COSA SUCCEDE:
        # La loss è un numero che rappresenta l'errore del modello
        # loss.backward() percorre la rete all'indietro e calcola:
        # - Quanto ogni peso ha contribuito all'errore finale
        # - In che direzione modificare ogni peso per ridurre l'errore
        #
        # ESEMPIO CONCRETO:
        # Immagina che la loss sia 5.2 (errore alto)
        # loss.backward() calcola per OGNI peso della rete:
        # - Peso1: +0.3 (aumentandolo, l'errore aumenta)
        # - Peso2: -0.7 (diminuendolo, l'errore diminuisce MOLTO)
        # - Peso3: -0.1 (diminuendolo, l'errore diminuisce POCO)
        #
        # COME LO FA?
        # Usa la backpropagation (regola della catena del calcolo differenziale)
        # PyTorch tiene traccia automaticamente di tutte le operazioni
        # Quando chiami backward, percorre il grafo delle operazioni al contrario
        #
        # RISULTATO:
        # Ogni parametro del modello (model.parameters()) avrà un attributo .grad
        # .grad contiene il gradiente (derivata) per quel parametro
        # I gradienti dicono all'ottimizzatore come aggiustare i pesi



        optimizer.step()
        # optimizer.step(): aggiorna i pesi usando i gradienti calcolati
        # Esegue: peso = peso - learning_rate * gradiente

        optimizer.zero_grad()
        # optimizer.zero_grad(): azzera i gradienti per il prossimo batch
        # Se non lo fai, i gradienti si accumulano batch dopo batch



        if batch % 100 == 0:
            # if batch % 100 == 0: ogni 100 batch stampa lo stato
            loss, current = loss.item(), (batch + 1) * len(X)
            # loss.item(): estrae il valore numerico della loss dal tensore
            # current: numero di campioni processati finora
            # (batch + 1) * len(X) = batch corrente * batch_size

            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
            # print: mostra la loss e quanti campioni processati
            # {loss:>7f}: loss formattata con 7 caratteri, virgola fissa
            # [{current:>5d}/{size:>5d}]: progresso tipo [ 1280/60000]

Calcolo dei paramentri in fase di TEST (dopo addestramento)

In [8]:
def test(dataloader, model, loss_fn):
    # test: funzione che valuta il modello sui dati di test
    # dataloader: contiene i dati di test in batch
    # model: la rete neurale da valutare
    # loss_fn: funzione per calcolare l'errore

    size = len(dataloader.dataset)
    # size: numero totale di campioni nel dataset di test (10.000 per Fashion-MNIST)

    num_batches = len(dataloader)
    # num_batches: numero totale di batch (size / batch_size)

    model.eval()
    # model.eval(): mette il modello in modalità valutazione
    # Disattiva dropout e batch normalization (comportamento diverso dal training)

    test_loss, correct = 0, 0
    # test_loss: accumulatore per la somma delle loss di tutti i batch
    # correct: accumulatore per il numero di predizioni corrette

    with torch.no_grad():
        # with torch.no_grad(): disabilita il calcolo dei gradienti
        # Durante il test non serve calcolare gradienti (risparmia memoria e velocizza)

        for X, y in dataloader:
            # Itera su tutti i batch nel dataloader di test
            X, y = X.to(device), y.to(device)
            # Sposta dati sul dispositivo (GPU/CPU)

            pred = model(X)
            # pred = model(X): passa le immagini al modello, ottieni predizioni
            # Le predizioni sono logits (valori grezzi per ogni classe)

            test_loss += loss_fn(pred, y).item()
            # test_loss +=: accumula la loss di questo batch
            # loss_fn(pred, y): calcola l'errore
            # .item(): estrae il valore numerico dal tensore

            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
            # pred.argmax(1): per ogni immagine, trova la classe con valore massimo
            # (pred.argmax(1) == y): confronta se la predizione è uguale all'etichetta vera
            # .type(torch.float): converte i booleani (True/False) in float (1.0/0.0)
            # .sum(): somma tutti i valori (conta quanti sono True)
            # .item(): estrae il numero intero dal tensore



    test_loss /= num_batches
    # test_loss /= num_batches: calcola la loss media dividendo per numero di batch

    correct /= size
    # correct /= size: calcola l'accuratezza dividendo predizioni corrette per totale campioni


    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")
    # print: mostra i risultati del test
    # Accuracy: percentuale di predizioni corrette (es. 85.3%)
    # Avg loss: loss media sul dataset di test

Blocco che fa partire l'addestramento: epoche, traing e test: fa 1 epoca con dati di training poi il test, per 5 volte questo ciclo

In [ ]:
epochs = 5
# epochs: numero di volte che l'algoritmo vedrà l'intero dataset
# Imposta a 5 epoche di addestramento

for t in range(epochs):
    # for t in range(epochs): ciclo che ripete per ogni epoca
    # t va da 0 a 4 (per epochs=5)

    print(f"Epoch {t+1}\n-------------------------------")
    # print: mostra il numero dell'epoca corrente
    # t+1 perché t parte da 0 ma vogliamo vedere da 1 a 5

    train(train_dataloader, model, loss_fn, optimizer)
    # train(): chiama la funzione di addestramento
    # Processa TUTTI i batch del training set (60.000 immagini)
    # Aggiorna i pesi del modello basandosi sull'errore

    test(test_dataloader, model, loss_fn)
    # test(): chiama la funzione di valutazione
    # Processa TUTTI i batch del test set (10.000 immagini)
    # Calcola accuratezza e loss senza aggiornare i pesi
    # Mostra come sta performando il modello su dati mai visti

print("Done!")
# print: messaggio finale quando l'addestramento è completato